# PREPROCESSING

In [15]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
import pandas as pd
from collections import Counter
from pyexpat import features
from scipy import stats
import os

# extra imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

## LOADING THE DATA

In [16]:
wifi = pd.read_csv('../data/raw/train_nt.csv', header=0, delimiter=';')
X_test = pd.read_csv('../data/raw/test_nolabels_nt.csv', header=0, delimiter=';')

print(f"Dataset shape: {wifi.shape}")
print(f"Missing values: {wifi.isnull().sum().sum()}")

Dataset shape: (12888, 262)
Missing values: 0


## Clean known Issues

### 1. Columns that don't provide information

In [17]:
columns_no_info = ['seq_ctrl', 'aoa', 'ID']

### 2. Constant columns

In [18]:
const_cols = [col for col in wifi.columns if wifi[col].nunique() <= 1]

In [19]:
columns_to_remove = columns_no_info + const_cols
print(columns_to_remove)

['seq_ctrl', 'aoa', 'ID', 'I0_1', 'Q0_1', 'I27_1', 'Q27_1', 'I28_1', 'Q28_1', 'I29_1', 'Q29_1', 'I30_1', 'Q30_1', 'I31_1', 'Q31_1', 'I32_1', 'Q32_1', 'I33_1', 'Q33_1', 'I34_1', 'Q34_1', 'I35_1', 'Q35_1', 'I36_1', 'Q36_1', 'I37_1', 'Q37_1', 'I0_2', 'Q0_2', 'I27_2', 'Q27_2', 'I28_2', 'Q28_2', 'I29_2', 'Q29_2', 'I30_2', 'Q30_2', 'I31_2', 'Q31_2', 'I32_2', 'Q32_2', 'I33_2', 'Q33_2', 'I34_2', 'Q34_2', 'I35_2', 'Q35_2', 'I36_2', 'Q36_2', 'I37_2', 'Q37_2']


In [20]:
def remove_columns(df, cols: list['str']) -> None:
    df.drop(columns=cols, inplace=True, errors = 'ignore')
    print(f"Removed columns: {cols}")


## Feature Engineering

In [21]:
def extract_iq_pairs(df):
    I1 = [f'I{n}_1' for n in range(64)]
    Q1 = [f'Q{n}_1' for n in range(64)]
    I2 = [f'I{n}_2' for n in range(64)]
    Q2 = [f'Q{n}_2' for n in range(64)]
    IQ1 = df[I1].values + 1j * df[Q1].values
    IQ2 = df[I2].values + 1j * df[Q2].values
    return IQ1, IQ2

def compute_amplitude(IQ):
    return np.abs(IQ)

def compute_phase(IQ):
    return np.angle(IQ)

def compute_phase_unwrapped(phase):
    return np.unwrap(phase, axis=1)

def sanitize_phase(phase_unwr):
    x = np.arange(phase_unwr.shape[1])
    sanitized = np.zeros_like(phase_unwr)
    for i in range(phase_unwr.shape[0]):
        m, c = np.polyfit(x, phase_unwr[i], 1)
        sanitized[i] = phase_unwr[i] - (m * x + c)
    return sanitized

def compute_frequency_domain_features(data):
    fft_result = np.fft.fft(data, axis=1)
    fft_mag = np.abs(fft_result)
    safe = fft_mag + 1e-10
    return {
        'spectral_energy': np.sum(fft_mag**2, axis=1),
        'spectral_entropy': stats.entropy(safe, axis=1),
        'spectral_centroid': np.sum(fft_mag * np.arange(fft_mag.shape[1]), axis=1) / np.sum(safe, axis=1)
    }

def compute_antenna_correlation(IQ1, IQ2):
    corr = []
    for i in range(len(IQ1)):
        c = np.corrcoef(np.abs(IQ1[i]), np.abs(IQ2[i]))[0, 1]
        corr.append(c)
    return np.nan_to_num(corr)

def compute_phase_consistency(p1, p2):
    diff = np.angle(np.exp(1j * (p1 - p2)))
    cos_m = np.mean(np.cos(diff), axis=1)
    sin_m = np.mean(np.sin(diff), axis=1)
    return np.sqrt(cos_m**2 + sin_m**2)

In [22]:
def extract_all_features(df):
    IQ1, IQ2 = extract_iq_pairs(df)

    amp1 = compute_amplitude(IQ1)
    amp2 = compute_amplitude(IQ2)
    ph1 = compute_phase(IQ1)
    ph2 = compute_phase(IQ2)

    ph1_un = compute_phase_unwrapped(ph1)
    ph2_un = compute_phase_unwrapped(ph2)

    san1 = sanitize_phase(ph1_un)
    san2 = sanitize_phase(ph2_un)

    rel_phase = san1 - san2

    features = {}

    # Frequency domain
    for k, v in compute_frequency_domain_features(amp1).items():
        features[f'freq_ant1_{k}'] = v
    for k, v in compute_frequency_domain_features(amp2).items():
        features[f'freq_ant2_{k}'] = v

    # Cross-antenna
    features['antenna_correlation'] = compute_antenna_correlation(IQ1, IQ2)
    features['phase_consistency'] = compute_phase_consistency(ph1, ph2)

    # Stats
    features['amp_ant1_mean'] = amp1.mean(axis=1)
    features['amp_ant1_std'] = amp1.std(axis=1)
    features['amp_ant2_mean'] = amp2.mean(axis=1)
    features['amp_ant2_std'] = amp2.std(axis=1)

    features['rel_phase_mean'] = rel_phase.mean(axis=1)
    features['rel_phase_std'] = rel_phase.std(axis=1)

    # AoA proxy
    aoa_est = rel_phase.mean(axis=1)
    features['aoa_sin'] = np.sin(aoa_est)
    features['aoa_cos'] = np.cos(aoa_est)

    return pd.DataFrame(features)

## Train / Val / Test Split

In [23]:
def build_dataset(df_raw):
    df_raw = df_raw.reset_index(drop=True)
    
    df_features = extract_all_features(df_raw)
    
    mag_cols = {}
    for antena in [1, 2]:
        for n in range(64):
            i_col = f"I{n}_{antena}"
            q_col = f"Q{n}_{antena}"
            mag_col = f"mag{n}_{antena}"
            mag_cols[mag_col] = np.sqrt(df_raw[i_col]**2 + df_raw[q_col]**2)
            
    df_magnitudes = pd.DataFrame(mag_cols)
    
    df_rssi = df_raw[['rssi1', 'rssi2']]
    
    X_clean = pd.concat([df_features, df_magnitudes, df_rssi], axis=1)
    
    return X_clean

In [24]:
X = wifi.drop(columns=['position'])
y = wifi['position']

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train = build_dataset(X_train_raw)
X_val   = build_dataset(X_val_raw)
X_test  = build_dataset(X_test)

remove_columns(X_train, columns_to_remove)
remove_columns(X_val, columns_to_remove)
remove_columns(X_test, columns_to_remove)

Removed columns: ['seq_ctrl', 'aoa', 'ID', 'I0_1', 'Q0_1', 'I27_1', 'Q27_1', 'I28_1', 'Q28_1', 'I29_1', 'Q29_1', 'I30_1', 'Q30_1', 'I31_1', 'Q31_1', 'I32_1', 'Q32_1', 'I33_1', 'Q33_1', 'I34_1', 'Q34_1', 'I35_1', 'Q35_1', 'I36_1', 'Q36_1', 'I37_1', 'Q37_1', 'I0_2', 'Q0_2', 'I27_2', 'Q27_2', 'I28_2', 'Q28_2', 'I29_2', 'Q29_2', 'I30_2', 'Q30_2', 'I31_2', 'Q31_2', 'I32_2', 'Q32_2', 'I33_2', 'Q33_2', 'I34_2', 'Q34_2', 'I35_2', 'Q35_2', 'I36_2', 'Q36_2', 'I37_2', 'Q37_2']
Removed columns: ['seq_ctrl', 'aoa', 'ID', 'I0_1', 'Q0_1', 'I27_1', 'Q27_1', 'I28_1', 'Q28_1', 'I29_1', 'Q29_1', 'I30_1', 'Q30_1', 'I31_1', 'Q31_1', 'I32_1', 'Q32_1', 'I33_1', 'Q33_1', 'I34_1', 'Q34_1', 'I35_1', 'Q35_1', 'I36_1', 'Q36_1', 'I37_1', 'Q37_1', 'I0_2', 'Q0_2', 'I27_2', 'Q27_2', 'I28_2', 'Q28_2', 'I29_2', 'Q29_2', 'I30_2', 'Q30_2', 'I31_2', 'Q31_2', 'I32_2', 'Q32_2', 'I33_2', 'Q33_2', 'I34_2', 'Q34_2', 'I35_2', 'Q35_2', 'I36_2', 'Q36_2', 'I37_2', 'Q37_2']
Removed columns: ['seq_ctrl', 'aoa', 'ID', 'I0_1', 'Q0_1',

## Preprocessing per model

In [25]:
cols = X_train.columns
PROCESSED_DATA_PATH = '../data/processed/'
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# 1 - non-scaled

X_train.to_csv(f'{PROCESSED_DATA_PATH}/X_train_non_scaled.csv', index=False)
X_val.to_csv(f'{PROCESSED_DATA_PATH}/X_val_non_scaled.csv', index=False)
X_test.to_csv(f'{PROCESSED_DATA_PATH}/X_test_non_scaled.csv', index=False)

# 2 - robust scaled
scaler = RobustScaler()
X_train_scaled_robust = scaler.fit_transform(X_train)
X_val_scaled_robust = scaler.transform(X_val)
X_test_scaled_robust = scaler.transform(X_test)

X_train_scaled_robust = pd.DataFrame(X_train_scaled_robust, columns=X_train.columns)
X_val_scaled_robust = pd.DataFrame(X_val_scaled_robust, columns=X_val.columns)
X_test_scaled_robust = pd.DataFrame(X_test_scaled_robust, columns=X_test.columns)

X_train_scaled_robust.to_csv(f'{PROCESSED_DATA_PATH}/X_train_robust_scaled.csv', index=False)
X_val_scaled_robust.to_csv(f'{PROCESSED_DATA_PATH}/X_val_robust_scaled.csv', index=False)
X_test_scaled_robust.to_csv(f'{PROCESSED_DATA_PATH}/X_test_robust_scaled.csv', index=False)

# 3 - min-max scaled    
scaler = MinMaxScaler()
X_train_scaled_minmax = scaler.fit_transform(X_train)
X_val_scaled_minmax = scaler.transform(X_val)
X_test_scaled_minmax = scaler.transform(X_test)

X_train_scaled_minmax = pd.DataFrame(X_train_scaled_minmax, columns=X_train.columns)
X_val_scaled_minmax = pd.DataFrame(X_val_scaled_minmax, columns=X_val.columns)
X_test_scaled_minmax = pd.DataFrame(X_test_scaled_minmax, columns=X_test.columns)

X_train_scaled_minmax.to_csv(f'{PROCESSED_DATA_PATH}/X_train_minmax_scaled.csv', index=False)
X_val_scaled_minmax.to_csv(f'{PROCESSED_DATA_PATH}/X_val_minmax_scaled.csv', index=False)
X_test_scaled_minmax.to_csv(f'{PROCESSED_DATA_PATH}/X_test_minmax_scaled.csv', index=False)

# 4 - standard scaled
scaler = StandardScaler()
X_train_scaled_standard = scaler.fit_transform(X_train)
X_val_scaled_standard = scaler.transform(X_val)
X_test_scaled_standard = scaler.transform(X_test)

X_train_scaled_standard = pd.DataFrame(X_train_scaled_standard, columns=X_train.columns)
X_val_scaled_standard = pd.DataFrame(X_val_scaled_standard, columns=X_val.columns)
X_test_scaled_standard = pd.DataFrame(X_test_scaled_standard, columns=X_test.columns)

X_train_scaled_standard.to_csv(f'{PROCESSED_DATA_PATH}/X_train_standard_scaled.csv', index=False)
X_val_scaled_standard.to_csv(f'{PROCESSED_DATA_PATH}/X_val_standard_scaled.csv', index=False)
X_test_scaled_standard.to_csv(f'{PROCESSED_DATA_PATH}/X_test_standard_scaled.csv', index=False)

# y 
y_train.to_csv(f'{PROCESSED_DATA_PATH}/y_train.csv', index=False)
y_val.to_csv(f'{PROCESSED_DATA_PATH}/y_val.csv', index=False)

print("\nPreprocessing completed for all versions.")


Preprocessing completed for all versions.
